In [1]:
from src.preprocessing import PreprocessConfig, preprocess_train_test
from src.features import FeatureConfig, engineer_train_test_features
import pandas as pd
import numpy as np

train_raw = pd.read_csv("D:/dataorg-financial-health-prediction-challenge20251204-19827-m2tn1n/Train.csv")
test_raw = pd.read_csv("D:/dataorg-financial-health-prediction-challenge20251204-19827-m2tn1n/Test.csv")

pre_cfg = PreprocessConfig(id_col="ID", target_col="Target")
feat_cfg = FeatureConfig(id_col="ID", target_col="Target")

# CatBoost path
train_clean, test_clean = preprocess_train_test(train_raw, test_raw, pre_cfg, for_model="catboost")
train_fe, test_fe = engineer_train_test_features(
    train_clean,
    test_clean,
    feat_cfg,
    collapse_rare_for_non_catboost=False
)

# Then:
y = train_fe["Target"]
X = train_fe.drop(columns=["Target"])

In [2]:
# For LightGBM

train_clean, test_clean = preprocess_train_test(train_raw, test_raw, pre_cfg, for_model="lightgbm")
train_fe, test_fe = engineer_train_test_features(
    train_clean,
    test_clean,
    feat_cfg,
    collapse_rare_for_non_catboost=True
)

In [3]:
print(train_fe.shape, test_fe.shape)
print(train_fe.columns.tolist())

(9618, 119) (2405, 118)
['ID', 'country', 'owner_age', 'attitude_stable_business_environment', 'attitude_worried_shutdown', 'compliance_income_tax', 'perception_insurance_doesnt_cover_losses', 'perception_cannot_afford_insurance', 'personal_income', 'business_expenses', 'business_turnover', 'business_age_years', 'motor_vehicle_insurance', 'has_mobile_money', 'current_problem_cash_flow', 'has_cellphone', 'owner_sex', 'offers_credit_to_customers', 'attitude_satisfied_with_achievement', 'has_credit_card', 'keeps_financial_records', 'perception_insurance_companies_dont_insure_businesses_like_yours', 'perception_insurance_important', 'has_insurance', 'covid_essential_service', 'attitude_more_successful_next_year', 'problem_sourcing_money', 'marketing_word_of_mouth', 'has_loan_account', 'has_internet_banking', 'has_debit_card', 'future_risk_theft_stock', 'business_age_months', 'medical_insurance', 'funeral_insurance', 'motivation_make_more_money', 'uses_friends_family_savings', 'uses_informa

In [5]:
# Inspect key engineered columns

check_cols = [
    "financial_access_score",
    "insurance_coverage_score",
    "resilience_risk_score",
    "formality_score",
    "business_confidence_score",
    "insurance_friction_score",
    "informal_coping_score",
    "margin_proxy",
    "expense_ratio",
    "turnover_to_expenses",
    "business_age_total_months",
    "business_age_bucket",
    "financial_health_proxy_score",
]

train_fe[check_cols].sample(50)

,financial_access_score,insurance_coverage_score,resilience_risk_score,formality_score,business_confidence_score,insurance_friction_score,informal_coping_score,margin_proxy,expense_ratio,turnover_to_expenses,business_age_total_months,business_age_bucket,financial_health_proxy_score
7873,0.0,0.0,2.0,2.0,3.0,3.0,0.0,80000.0,0.733331,1.363630,144.0,y10_plus,3.0
8869,0.0,0.0,2.0,0.0,2.0,1.0,0.0,-1500.0,1.998668,0.499833,24.0,y0_5_to_2,0.0
4102,0.0,0.0,0.0,0.0,2.0,3.0,0.0,600.0,0.166436,5.950413,36.0,y2_to_5,2.0
919,0.0,1.0,0.0,0.0,2.0,0.0,0.0,1750.0,0.299880,3.328895,216.0,y10_plus,3.0
5677,0.0,0.0,0.0,0.0,2.0,1.0,0.0,403.0,0.193613,5.102041,96.0,y5_to_10,2.0
594,0.0,0.0,1.0,1.0,2.0,2.0,0.0,5000.0,0.499950,1.999600,143.0,y10_plus,2.0
3508,0.0,0.0,0.0,1.0,2.0,2.0,0.0,1200.0,0.499792,1.998335,36.0,y2_to_5,3.0
146,0.0,0.0,1.0,1.0,3.0,0.0,0.0,120000.0,0.666665,1.499994,156.0,y10_plus,3.0
3833,0.0,0.0,0.0,0.0,0.0,3.0,0.0,NaN,NaN,NaN,77.0,y5_to_10,0.0
8989,0.0,0.0,0.0,0.0,3.0,0.0,0.0,1500.0,0.399840,2.497502,420.0,y10_plus,3.0


# Feature Engineering Aligned with FHI Dimensions

To improve both model performance and interpretability, I formalized feature engineering into a reusable module (`src/features.py`) aligned with the conceptual design of the Financial Health Index (FHI).

The engineered features were grouped into several domain-driven dimensions:

## 1. Financial Access
A composite score was built from indicators of formal and digital financial inclusion, including:
- mobile money
- debit card
- credit card
- internet banking
- loan account access

## 2. Insurance Coverage
A second score captured financial protection through:
- insurance ownership
- medical insurance
- funeral insurance
- motor vehicle insurance

## 3. Resilience and Financial Stress
Risk-related indicators were combined to measure business vulnerability:
- cash flow problems
- sourcing money difficulties
- worry about shutdown
- theft risk

## 4. Formality
Indicators of business formalization were aggregated:
- tax compliance
- record keeping

## 5. Business Confidence and Outlook
Attitudinal measures were combined to reflect optimism and business confidence.

## 6. Informal Coping and Insurance Friction
Additional scores captured dependence on informal finance and perceived barriers to insurance access.

## 7. Business Health Ratios
Numerical ratio features such as margin proxies and expense ratios were created from turnover, expenses, and income variables.

## 8. Contextual Interaction Features
Because earlier validation showed strong country-driven effects, several country- and gender-based interaction features were added.

This feature engineering layer transformed raw survey variables into economically meaningful representations that improved both model quality and the interpretability of the final pipeline.